# <center> Assignment 1 — Netflix Trends </center>

**Problem Statement:** Examing how the number of titles added to Netflix changed over time (by year). Also observing which countries contribute the most titles over the year.

**Reproducibility rule:** `Kernel → Restart & Run All` should work without errors.

**Workflow:** Imports → Data loading → Checks → Analysis & Visuals → Interpretation.

## Imports
Importing numpy, pandas, and matplotlib

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Loading
Loading the dataset

In [ ]:
# loading dataset 
DATA_PATH = "../data/netflix_titles.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## Data Dictionary
- **show_id** - unique identifier for each title
- **type** - Movie or TV show
- **title** - name of the show or movie
- **director** - name of the director
- **cast** - names of the actors
- **country** - country where it was made
- **date_added** - the date when the movie was added
- **release_year** - year when it was released
- **rating** - minimum age category to watch the title
- **duration** - runtime of the show or movie
- **listed_in** - genre it is listed in
- **description** - breif overview of what is in the show

## Data Cleaning
Checking if data is clean and cleaning it if it is not

In [ ]:
# Checking for Duplicates and removing if any
if df[df.duplicated()].size != 0:
    df = df.drop_duplicates()

In [ ]:
# checking number of NAs
df.isna().sum()

In [ ]:
# fixing datatypes
df["date_added"] = pd.to_datetime(df["date_added"])
df["release_year"] = df["release_year"].astype(int)

In [ ]:
# checking if type column has only Movie and TV show as values
df["type"].value_counts()

### Data Checks

In [ ]:
# Checking shape of data
df.shape

In [ ]:
df.describe()

# <center>Analysis & Visuals</center>

## Identifying Outliers
Checking to see what durations of movies are shorter or longer than usual

In [ ]:
# grouping movies
movies = df[df["type"] == "Movie"]

In [ ]:
# dropping movies with NA as duration
movies = movies.dropna(subset ="duration")

In [ ]:
# Converting movie duration from string to float
movies["duration"] = movies["duration"].str.extract(r"(\d+)").astype(float)

In [ ]:
# making a numpy array for movies
movieDuration = movies["duration"].to_numpy()
movieDuration

In [ ]:
# function to find outliers by taking the lowest 10% and highest 10% of the values
def find_outliers(arr):
    lowerBound = np.percentile(arr, 10)
    upperBound = np.percentile(arr, 90)

    outliers = (arr < lowerBound) | (arr > upperBound)

    return outliers

In [ ]:
# calling function and printing the outliers out
outliers = find_outliers(movieDuration)

movies[outliers]

## Comparison of TV shows and Movies added to Netflix per year
Comparing the number of TV shows to the number of Movies added to Netflix each year

In [ ]:
# takes the dataframe as input and returns a table showing how many Movies, TV Shows, and total titles were added to Netflix each year.

def titles_added_per_year(dataframe):
    dataframe["year_added"] = dataframe["date_added"].dt.year
    summary = (
        dataframe
        .dropna(subset=["year_added"])   # dropping NA in year_added column
        .groupby(["year_added", "type"])   # grouping by year_added
        .size()
        .unstack(fill_value=0)
    )

    summary["Total"] = summary.sum(axis=1) # adding total released

    return summary

In [ ]:
# Calling function and printing out table
titlesAddedPerYear = titles_added_per_year(df)
titlesAddedPerYear

In [ ]:
# Plotting a line graph for the number of titles added to Netflix per year
titlesAddedPerYear.plot()
plt.xlabel("Year")
plt.ylabel("Number of titles added")
plt.title("Titles Added to Netflix Per Year")
plt.show()

##### **Plot Description:**
This plot shows the number of Titles added to Netflix each year while also showing a comparison between Movies and TV shows added. In the beginning, they are adding titles at a steady rate, however, in 2016 there is a huge leap in the number and this trend continues. The number of TV shows added show an upwards trend but are significantly lower in number than the movies added at the same time. In 2019, both hit their peak and the number of titles added are slowly decreasing with movies dropping more in number than TV shows.

## Finding out which Country contributes the most titles to Netflix
Grouping data by country and finding out which country has the most number of Netflix titles.

In [ ]:
# Returns a table with the number of Movies, TV Shows, and total titles grouped together by country.
def titles_by_country(dataframe):
    countryRows = (
        dataframe
        .dropna(subset=["country"])
        .assign(country=dataframe["country"].str.split(", "))
        .explode("country")
    )

    summary = (
        countryRows
        .groupby(["country", "type"])
        .size()
        .unstack(fill_value=0)
    )

    summary["Total"] = summary.sum(axis=1)

    return summary.sort_values("Total", ascending=False)

In [ ]:
country_summary = titles_by_country(df)
country_summary

In [ ]:
# Plotting a bar graph for the top 5 countries with the most Netflix titles
country_summary.head().plot.bar()
plt.xlabel("Country")
plt.ylabel("Number of Titles")
plt.title("Top 5 Countries by number of Netflix titles")
plt.show()

##### **Plot Description:**
This bar graph shows the number of titles added by the top 5 countries. United States leads the chart with the most titles of all time and almost as many TV Shows as the next country has Movies. Overall each country has fewer TV shows than movies. The top 5 countries are United States, India, United Kingdom, Canada, and France.

# <center>Interpretations</center>

### Q- How has the number of titles added to Netflix changed over time (by year)?

Ans- The number of titles added to Netflix were on a slow rise upto 2016, where it started to plummet and peaked in 2019. After 2019 the number of titles added are on a slow decline.

### Q- Which countries contribute the most titles?

Ans- The top 5 countries which contributed to the most titles on Netflix are the United States, India, United Kingdom, Canada, and France.